# **Multi-output Regression Use Case Demonstration**



**Given the `sqft` and `lotsize` (2 input features) predict the `price` and `bedrooms` (2 output features) of a house**

In [1]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization
from keras.initializers import he_normal
from keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler

In [2]:
# This function generates some synthetic data {sqft, bedrooms, lotsize and price}
def generate_data(count, noutputs=1):
    sqft_ranges = [0, 1500, 2500, 3500, 10000]
    price_ranges = [800000, 1000000, 2500000, 3000000]

    x = []
    y = []

    for i in range(count):
        sqft = random.randint(1800,4500)
        nbedrooms = 1+ int(sqft/1000)
        lotsize = int(random.uniform(1.2, 2.0) * sqft)

        price = 0
        for r in range(len(sqft_ranges)-1):
            if sqft >= sqft_ranges[r] and sqft < sqft_ranges[r+1]:
                price = price_ranges[r]
                break
        assert price != 0

        lotsize_value = int(random.randint(50000, 300000) * (lotsize/sqft))
        price = price + lotsize_value
        price = round(price, -3)

        if noutputs == 1:
            x.append([sqft, nbedrooms, lotsize])
            y.append(price)
        else:
            x.append([sqft, lotsize])
            y.append([price, nbedrooms])

    return x, y

**Let's initialize a Standardization module we'll use in our model**

In [3]:
norm_x = StandardScaler() # We'll use this to standardize our input
norm_y = StandardScaler() # We'll use this to standardize our output

**Let's `build` a model to predict 2 outputs**

In [4]:
def build_model_for_two_outputs():
    model = Sequential()

    # Hidden Layer(s)
    model.add(Dense(10, input_dim=2, kernel_initializer=he_normal(), activation="relu"))

    # Output Layer
    model.add(Dense(2))

    model.compile(optimizer=Adam(learning_rate=0.01), loss='mse')
    return model

model2 = build_model_for_two_outputs()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Let's generate some data**

In [5]:

# Generate some house price data
x, y = generate_data(10000, noutputs=2)

# Let's display the data we generated
data = pd.DataFrame(
    {'X_sqft': [rec[0] for rec in x],
    'X_lotsize': [rec[1] for rec in x],
    'Y_price': [rec[0] for rec in y],
    'Y_nbedrooms': [rec[1] for rec in y]})
data

,X_sqft,X_lotsize,Y_price,Y_nbedrooms
0,3448,6435,2797000,4
1,2181,3682,1087000,3
2,3804,6838,3394000,4
3,2166,4155,1456000,3
4,3369,4359,2623000,4
...,...,...,...,...
9995,2424,4152,1225000,3
9996,3322,5044,2672000,4
9997,4195,5738,3161000,5
9998,1943,3003,1340000,2


**Let's `train` our model**

In [6]:
def train_model_for_two_outputs(model, x, y):
    # Normalize data
    xnorm = norm_x.fit_transform(np.array(x))
    ynorm = norm_y.fit_transform(np.array(y))

    # Train the model
    model.fit(xnorm, ynorm, epochs=100, batch_size=10, verbose=1)

train_model_for_two_outputs(model2, x, y)

Epoch 1/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 0.1640
Epoch 2/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - loss: 0.1251
Epoch 3/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.1163
Epoch 4/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - loss: 0.1126
Epoch 5/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.1026
Epoch 6/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 0.0816
Epoch 7/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0725
Epoch 8/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0678
Epoch 9/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0647
Epoch 10/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - loss: 0.0642
Epoch 11/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0613
Epoch 12/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0611
Epoch 13/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0599
Epoch 14/100
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0590
Epoch 15/100
10

**Let's use our model to make predictions**

In [7]:
def predict_model_for_two_outputs(model):
    x, yt = generate_data(20, noutputs=2)

    xnorm = norm_x.transform(np.array(x))

    predictions = model.predict(xnorm)

    # De-Normalize predictions
    yp = norm_y.inverse_transform(predictions)

    return x, yt, yp

x, yt, yp = predict_model_for_two_outputs(model2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step


**Let's see what our predictions look like**

In [8]:
def show_predictions_for_two_outputs(x, yt, yp):
    price_offby = []
    price_percent = []

    nbedrooms_offby = []
    nbedrooms_percent = []

    for i in range(len(yp)):
        price_diff = int(yt[i][0] - yp[i][0])
        price_offby.append(price_diff)
        price_percent.append(int(100*(price_diff/yt[i][0])))

        nbedrooms_diff = yt[i][1] - yp[i][1]
        nbedrooms_offby.append(nbedrooms_diff)
        nbedrooms_percent.append(int(100*(nbedrooms_diff/yt[i][1])))

    data = pd.DataFrame(
            {'sqft': [rec[0] for rec in x],
            'lotsize': [rec[1] for rec in x],
            'price': [int(p[0]) for p in yt],
            'price_pred': [int(p[0]) for p in yp],
            'price_offby': price_offby,
            'price_percent_off': price_percent,
            'nbedrooms': [p[1] for p in yt],
            'nbedrooms_pred': [int(p[1]) for p in yp],
            'nbedrooms_offby': nbedrooms_offby,
            'nbedrooms_percent_of': nbedrooms_percent})
    return data

show_predictions_for_two_outputs(x, yt, yp)

,sqft,lotsize,price,price_pred,price_offby,price_percent_off,nbedrooms,nbedrooms_pred,nbedrooms_offby,nbedrooms_percent_of
0,1801,3245,1362000,1395310,-33310,-2,2,1,0.399834,19
1,2643,3612,2682000,2741130,-59130,-2,3,3,-0.043835,-1
2,3299,5015,2927000,2746638,180362,6,4,4,-0.075865,-1
3,3809,7587,3211000,3365266,-154266,-4,4,4,-0.292426,-7
4,2532,4668,2800000,2649658,150341,5,3,3,-0.050011,-1
5,1898,3545,1464000,1395245,68754,4,2,2,-0.035197,-1
6,3178,5331,2791000,2762625,28374,1,4,4,-0.046151,-1
7,3558,4817,3120000,3056963,63036,2,4,4,-0.051283,-1
8,3592,4722,3243000,3112375,130624,4,4,4,-0.043125,-1
9,2556,5104,3066000,2812120,253880,8,3,3,-0.060996,-2
